In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))


# ── shared/mlfp05/ex_2.py ──
"""
Shared utilities for MLFP05 Exercise 2 — CNNs on CIFAR-10.

Common infrastructure used by all technique files:
  - CIFAR-10 data loading and normalisation
  - Device detection and precision settings
  - ExperimentTracker / ModelRegistry setup
  - LightningModule wrapper for training
  - Visualisation helpers
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pytorch_lightning as pl
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.engines.model_registry import LocalFileArtifactStore
from kailash_ml.types import MetricSpec

# ── Environment & Reproducibility ────────────────────────────────────
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42, workers=True)

DEVICE = get_device()
# Notebook-safe path resolution: __file__ is undefined when this module is
# inlined into a Colab notebook by scripts/generate_selfcontained_notebook.py.
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
    ARTIFACT_DIR = _HERE.parent
except NameError:
    REPO_ROOT = Path.cwd()
    ARTIFACT_DIR = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"

# The ModelRegistry stores artifact files (model.pkl, model.onnx) in an
# ArtifactStore. We construct the store explicitly so exercises can attach
# an ONNX serving artifact to a registered version (see attach_onnx_artifact).
ARTIFACT_STORE = LocalFileArtifactStore(".kailash_ml/artifacts")

N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

# ── CIFAR-10 class labels (used for visualisation and apply sections) ─
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Per-channel normalisation statistics (CIFAR-10 population)
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


# ═══════════════════════════════════════════════════════════════════════
# Data Loading
# ═══════════════════════════════════════════════════════════════════════
def load_cifar10() -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    DataLoader,
    DataLoader,
]:
    """Load and normalise the full CIFAR-10 dataset.

    Returns:
        X_train, y_train, X_val, y_val, train_loader, val_loader
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_train = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_train = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))],
        dtype=torch.long,
    )
    X_val = torch.stack([val_set[i][0] for i in range(len(val_set))])
    y_val = torch.tensor(
        [val_set[i][1] for i in range(len(val_set))],
        dtype=torch.long,
    )

    # Per-channel normalisation
    X_train = (X_train - CIFAR_MEAN) / CIFAR_STD
    X_val = (X_val - CIFAR_MEAN) / CIFAR_STD

    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    print(
        f"CIFAR-10: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, "
        f"classes={N_CLASSES}: {CLASS_NAMES}"
    )
    return X_train, y_train, X_val, y_val, train_loader, val_loader


# ═══════════════════════════════════════════════════════════════════════
# Kailash Engine Setup
# ═══════════════════════════════════════════════════════════════════════
async def setup_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry, has_registry
    """
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker and
    # ModelRegistry use incompatible _kml_model_versions schemas. Route them
    # to separate sqlite files until upstream fixes the conflict.
    db = f"sqlite:///{db_name}"
    registry_db_name = (
        db_name.replace(".db", "_registry.db")
        if db_name.endswith(".db")
        else db_name + "_registry.db"
    )
    registry_db = f"sqlite:///{registry_db_name}"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn, artifact_store=ARTIFACT_STORE)
    return conn, tracker, "m5_cnns", registry, True


def init_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Sync wrapper for setup_engines."""
    return asyncio.run(setup_engines(db_name))


# ═══════════════════════════════════════════════════════════════════════
# Lightning Training Infrastructure
# ═══════════════════════════════════════════════════════════════════════
class LitCNN(pl.LightningModule):
    """Lightning wrapper for any nn.Module classifier.

    Tracks per-epoch training loss and validation accuracy for later
    logging to ExperimentTracker and visualisation with ModelVisualizer.
    """

    def __init__(self, model: nn.Module, lr: float = 1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.train_losses: list[float] = []
        self.val_accs: list[float] = []
        self._batch_losses: list[float] = []
        self._val_correct = 0
        self._val_total = 0

    def training_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        loss = F.cross_entropy(logits, y)
        self._batch_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self._batch_losses:
            self.train_losses.append(float(np.mean(self._batch_losses)))
            self._batch_losses = []

    def validation_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        self._val_correct += int((logits.argmax(dim=-1) == y).sum().item())
        self._val_total += int(y.size(0))

    def on_validation_epoch_end(self):
        if self._val_total > 0:
            self.val_accs.append(self._val_correct / self._val_total)
            self._val_correct = 0
            self._val_total = 0

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)


def get_precision_setting() -> str:
    """Return the optimal Lightning precision string for the current backend.

    Routes through ``kailash_ml.device()`` so MPS / CUDA / ROCm / Intel XPU /
    CPU are all selected by the same policy the rest of the platform uses.
    BackendInfo.precision is the canonical answer ("16-mixed" on Apple MPS
    + Tensor-Core GPUs, "32" on older GPUs and CPU).
    """
    import kailash_ml as km

    backend = km.device()
    print(f"  {backend.backend} detected -- using {backend.precision} precision")
    return backend.precision


def get_accelerator() -> str:
    """Return the Lightning accelerator string for the current backend."""
    import kailash_ml as km

    return km.device().accelerator


PRECISION = get_precision_setting()
ACCELERATOR = get_accelerator()


async def train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Train a CNN with Lightning and log metrics to ExperimentTracker.

    Uses the kailash-ml 1.1.1 ``tracker.track(...)`` async context manager.
    On normal exit the run is marked FINISHED; on exception it is marked FAILED.
    """
    lit = LitCNN(model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "architecture": name,
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": str(BATCH_SIZE),
                "dataset_size": str(len(train_loader.dataset)),
                "precision": PRECISION,
                "accelerator": ACCELERATOR,
            }
        )

        trainer.fit(lit, train_loader, val_loader)

        for epoch_idx, loss in enumerate(lit.train_losses):
            await run.log_metric("train_loss", loss, step=epoch_idx + 1)
        for epoch_idx, acc in enumerate(lit.val_accs):
            await run.log_metric("val_accuracy", acc, step=epoch_idx + 1)

        await run.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    print(
        f"  [{name}] final train loss={lit.train_losses[-1]:.4f}  "
        f"val acc={lit.val_accs[-1]:.3f}"
    )
    return lit.train_losses, lit.val_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            lr,
            epochs,
        )
    )


# ═══════════════════════════════════════════════════════════════════════
# Model Registration
# ═══════════════════════════════════════════════════════════════════════
async def register_model_async(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Register a trained model with metrics in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="val_accuracy", value=final_acc),
            MetricSpec(name="epochs", value=float(epochs)),
            MetricSpec(name="batch_size", value=float(BATCH_SIZE)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={final_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Sync wrapper for register_model_async."""
    return asyncio.run(
        register_model_async(registry, name, model, final_loss, final_acc, epochs)
    )


# ═══════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ═══════════════════════════════════════════════════════════════════════
def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance.

    Delegates to the canonical factory in shared.mlfp05 — the P2
    experimental notice is acknowledged there.
    """

    return _canonical()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
    x_label: str = "Epoch",
    y_label: str = "Value",
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(
        metrics=metrics,
        x_label=x_label,
        y_label=y_label,
    )
    fig.write_html(str(output_path))
    print(f"  Saved plot: {output_path}")


def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters())


def denormalise_cifar(img_tensor: torch.Tensor) -> torch.Tensor:
    """Reverse CIFAR-10 normalisation for display.

    Args:
        img_tensor: shape (C, H, W) or (B, C, H, W), normalised

    Returns:
        Tensor with pixel values clipped to [0, 1]
    """
    if img_tensor.dim() == 3:
        mean = CIFAR_MEAN.squeeze(0)
        std = CIFAR_STD.squeeze(0)
    else:
        mean = CIFAR_MEAN
        std = CIFAR_STD
    return (img_tensor * std + mean).clamp(0, 1)


# ═══════════════════════════════════════════════════════════════════════
# Serving Helpers (ONNX export + InferenceServer)
# ═══════════════════════════════════════════════════════════════════════
IMAGE_SHAPE = (3, 32, 32)
N_PIXELS = 3 * 32 * 32


class FlatImageAdapter(nn.Module):
    """Wrap an image classifier so it accepts flat pixel rows.

    InferenceServer's ONNX runtime turns each request record into ONE row of
    a 2-D float array, so the exported graph must take ``(batch, 3072)``.
    This adapter reshapes rows back to ``(batch, 3, 32, 32)`` before calling
    the wrapped CNN. ``predict(X)`` is the method ``OnnxBridge.validate``
    calls to get the native (PyTorch) outputs for comparison.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x.reshape(-1, *IMAGE_SHAPE))

    def predict(self, X) -> np.ndarray:
        device = next(self.model.parameters()).device
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
            return self.forward(x).cpu().numpy()


def images_to_records(images: torch.Tensor) -> list[dict[str, float]]:
    """Turn a batch of images into InferenceServer request records.

    One record per image, one key per pixel. Keys are zero-padded so the
    record order is the pixel order the ONNX graph expects.
    """
    rows = images.reshape(len(images), -1).cpu().tolist()
    return [{f"px{j:04d}": float(v) for j, v in enumerate(row)} for row in rows]


def attach_onnx_artifact(name: str, version: int, onnx_path: Path) -> None:
    """Store an exported ONNX file as ``model.onnx`` for a registered version.

    ``ModelRegistry.register_model`` saves the training artifact as
    ``model.pkl``; ``InferenceServer.from_registry(..., runtime="onnx")``
    loads ``model.onnx`` from the same ArtifactStore.

    OnnxBridge writes large weights to a sibling ``<file>.onnx.data`` file
    (ONNX external data); storing only the ``.onnx`` bytes would serve a model
    without its weights, so it is stored as ONE self-contained protobuf.
    """
    import onnx

    model_proto = onnx.load(str(onnx_path))  # pulls in any .onnx.data weights
    asyncio.run(
        ARTIFACT_STORE.save(name, version, model_proto.SerializeToString(), "model.onnx")
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 2.2 — ResNet + Squeeze-and-Excitation: Depth and Attention
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain WHY deeper networks fail (vanishing gradients, degradation
#     problem) in terms a non-technical manager can understand
#   - Build a residual block (y = F(x) + x) and explain how skip
#     connections keep gradients healthy
#   - Implement Squeeze-and-Excitation (SE) channel attention — "which
#     feature maps matter for THIS specific input?"
#   - Compare ResNetSE against SimpleCNN on the same dataset with
#     ExperimentTracker metrics
#   - Visualise Grad-CAM heatmaps showing WHERE the model looks to make
#     each decision (not just what it predicts, but WHY)
#   - Apply this to manufacturing quality inspection at a Singapore
#     semiconductor fab
#
# PREREQUISITES: M5/ex_2/01_simple_cnn.py (SimpleCNN training, filter
#   visualisation, ExperimentTracker basics)
# ESTIMATED TIME: ~40 min
#
# DATASET: CIFAR-10 — same 50K training images as 01_simple_cnn.py.
#   We re-use the data loading from helpers.py.
#
# PHASES:
#   1. THEORY  — Why depth fails and how residuals + attention fix it
#   2. BUILD   — ResBlock, SEBlock, and ResNetSE architecture
#   3. TRAIN   — Train and compare against SimpleCNN
#   4. VISUALISE — Grad-CAM heatmaps showing model attention
#   5. APPLY   — Semiconductor wafer inspection at a Singapore fab
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt



## PHASE 1 — THEORY: Why Depth Fails, How Residuals Fix It


THE DEGRADATION PROBLEM:
    Deeper plain networks can reach HIGHER training error than shallower
    ones — an optimisation difficulty, not overfitting (He et al. 2015).

  RESIDUAL FIX: y = F(x) + x
    The network only learns what to ADD (the residual); the identity is
    free (F(x) = 0), and gradients also get a direct path through the skip.

  SE ATTENTION: "Which feature maps matter for THIS image?"
    Squeeze (global avg pool) -> Excite (small MLP) -> Scale (re-weight)
    Adds <1% parameters, measurable accuracy improvement.


In [ ]:
# THE DEGRADATION PROBLEM (discovered by He et al., 2015):
#
# Intuition for a non-technical audience:
#   Imagine a game of "telephone" with 50 people in a chain. Each
#   person whispers the message to the next. By the time it reaches
#   person #50, the message is garbled beyond recognition.
#
#   A deep "plain" network (no skip connections) has the same
#   weakness: every layer must re-transmit everything useful, so even
#   copying the input through unchanged (the identity) has to be
#   LEARNED by a stack of non-linear layers — and optimisers find that
#   hard. He et al. (2015, §4.1) note this is UNLIKELY to be vanishing
#   gradients: their plain nets used BatchNorm and the gradients were
#   healthy. The deeper plain nets were simply harder to optimise and
#   reached HIGHER training error than shallower ones.
#
# THE RESIDUAL FIX:
#   Instead of each person paraphrasing the message, you give person
#   #1 a direct phone line to person #50. Now person #50 can hear
#   BOTH the original message AND the chain's version. If the chain
#   garbled it, the original is still available.
#
#   Mathematically: y = F(x) + x
#   - F(x) is the "chain's version" (what the conv layers compute)
#   - x is the "direct phone line" (the skip connection)
#   - The network only needs to learn the RESIDUAL: F(x) = y - x
#     (what to ADD to the input, not the entire transformation)
#
# WHY THIS MATTERS IN PRACTICE:
#   Before ResNet (2015): on ImageNet, a 34-layer plain network had
#   HIGHER training error than an 18-layer one. Not overfitting — an
#   optimisation difficulty.
#
#   After ResNet: 152-layer networks outperformed 20-layer networks.
#   ResNet won the 2015 ImageNet competition with 3.57% top-5 error
#   (human performance: ~5.1%).
#
# SQUEEZE-AND-EXCITATION (SE) ATTENTION:
#   After residual blocks extract features, SE blocks ask: "which of
#   these 32 feature maps are most useful for THIS specific image?"
#
#   Analogy: A doctor examining an X-ray doesn't weigh all visual
#   features equally. For a lung scan, they focus on tissue density
#   patterns. For a bone scan, they focus on edge sharpness. SE blocks
#   learn this "selective attention" automatically.
#
#   Mechanism:
#     1. SQUEEZE: Global average pool each feature map to one number
#        (32 feature maps -> 32 numbers summarising "how active is
#        this filter?")
#     2. EXCITE: A small MLP learns to produce a weight for each
#        channel (32 numbers -> 32 weights between 0 and 1)
#     3. SCALE: Multiply each feature map by its weight (amplify
#        useful channels, suppress irrelevant ones)
#
#   Cost: <1% extra parameters. Benefit: measurable accuracy boost.

print("=" * 70)
print("  PHASE 1 — THEORY: Residual Learning + Channel Attention")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: ResBlock, SEBlock, and ResNetSE


Two conv layers with BatchNorm and ReLU. The skip connection adds
    the input directly to the output, giving gradients a highway through
    the network.


Squeeze: global average pool -> one number per channel
    Excite: small MLP -> one weight per channel (0 to 1 via sigmoid)
    Scale: multiply each feature map by its learned weight


Architecture:
        Stem: Conv(3->32) -> BN -> ReLU -> MaxPool(2)   [32x32 -> 16x16]
        ResBlock(32) -> SEBlock(32) -> ResBlock(32)
        AdaptiveAvgPool -> Linear(32 -> 10)


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: ResBlock + SEBlock + ResNetSE")
print("=" * 70)


class ResBlock(nn.Module):

    def __init__(self, channels: int):
        super().__init__()
        # TODO: Build residual block — two 3x3 convs that keep the channel
        #   count AND the spatial size ("same" padding), each with its own
        #   batch norm, so the output can be added back to the input
        self.conv1 = ____
        self.bn1 = ____
        self.conv2 = ____
        self.bn2 = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement residual forward pass (identity already saved below)
        #   1. First conv -> its batch norm -> ReLU
        #   2. Second conv -> its batch norm (NO activation yet)
        #   3. Add the saved identity, THEN apply ReLU  <-- the skip connection
        identity = x
        out = ____
        out = ____
        return ____


class SEBlock(nn.Module):

    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        # TODO: Build the SE MLP — a bottleneck: channels -> hidden (ReLU)
        #   -> back to channels, ending in a sigmoid so each channel gets a
        #   weight in (0, 1)
        self.fc = nn.Sequential(
            ____,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        # TODO: Implement squeeze-excite-scale
        #   1. Squeeze s: global average pool over H and W -> shape (b, c),
        #      one number per channel
        #   2. Excite w: run s through self.fc, reshape to (b, c, 1, 1) so it
        #      broadcasts over the spatial dims -> per-channel weights
        #   3. Scale: re-weight each feature map of x by its channel weight
        s = ____
        w = ____
        return ____


class ResNetSE(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        # TODO: Build the stem — 3x3 conv 3 -> 32 channels ("same" padding)
        #   -> batch norm -> ReLU -> 2x2 max-pool [spatial: 32 -> 16]
        self.stem = nn.Sequential(
            ____,
        )
        # TODO: Wire up ResBlock -> SEBlock -> ResBlock -> pool -> classifier
        #   All blocks work on the stem's 32 channels; pool = global average
        #   pool to 1x1; classifier maps 32 features to n_classes logits
        self.block1 = ____
        self.se1 = ____
        self.block2 = ____
        self.pool = ____
        self.fc = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Forward pass — apply block1, se1 and block2 in that order
        x = self.stem(x)
        x = ____
        x = ____
        x = ____
        x = self.pool(x).flatten(1)
        return self.fc(x)



### Checkpoint 1: Architecture builds correctly


In [ ]:
resnet_test = ResNetSE()
dummy_input = torch.randn(2, 3, 32, 32)
dummy_output = resnet_test(dummy_input)
assert dummy_output.shape == (
    2,
    N_CLASSES,
), f"ResNetSE output should be (batch, {N_CLASSES}), got {dummy_output.shape}"

resnet_params = count_parameters(resnet_test)
print(f"\nResNetSE built successfully")
print(f"  Parameters: {resnet_params:,}")
print(f"  Components: stem + ResBlock + SEBlock + ResBlock + classifier")
print(f"  Skip connections: 2 (one per ResBlock)")
print(f"  SE reduction ratio: 8 (32 channels -> 4 hidden -> 32 weights)")

# INTERPRETATION: ResNetSE has more parameters than SimpleCNN due to the
# extra ResBlock convolutions and SE MLP, but the skip connections make
# gradient flow more efficient, so the network trains faster per-parameter.
# The SE block adds only ~200 parameters but can measurably improve accuracy.
print("\n--- Checkpoint 1 passed --- ResNetSE architecture verified\n")
del resnet_test, dummy_input, dummy_output



## PHASE 3 — TRAIN: ResNetSE vs SimpleCNN Comparison


In [ ]:
print("=" * 70)
print("  PHASE 3 — TRAIN: ResNetSE on full CIFAR-10")
print("=" * 70)

# Load data
X_train, y_train, X_val, y_val, train_loader, val_loader = load_cifar10()

# Set up engines
conn, tracker, exp_name, registry, has_registry = init_engines()

# TODO: Train ResNetSE
#   1. resnet_se: a fresh ResNetSE
#   2. Train it with train_model (run name "ResNetSE", same tracker,
#      experiment, loaders and EPOCHS as SimpleCNN in 01)
print(f"\nTraining ResNetSE for {EPOCHS} epochs on {X_train.shape[0]:,} images...")
resnet_se = ____
resnet_losses, resnet_accs = ____



## DIAGNOSTIC CHECKPOINT — five instruments (residual connections)


Cross-entropy on one (images, labels) batch, on the model's device.


In [ ]:
# ResNetSE = residual blocks + squeeze-and-excitation. Skip connections
# give every block a direct gradient path, so compare this pad's
# gradient-flow reading with the plain stacks you have seen (ex_1/07).
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _ce_loss(m, batch):
    xb, yb = batch
    dev = next(m.parameters()).device
    return F.cross_entropy(m(xb.to(dev)), yb.to(dev))


print("\n── Diagnostic Report (ResNetSE) ──")
diag, findings = run_diagnostic_checkpoint(
    resnet_se,
    train_loader,
    _ce_loss,
    title="ResNetSE (CIFAR-10)",
    train_losses=resnet_losses,
    show=False,
)
print_prescription_pad(findings, "ResNetSE (CIFAR-10)")

# ══════ READING THE PRESCRIPTION PAD (key: see ex_1/01_standard_ae.py) ══════
# SE blocks rescale each channel by a sigmoid weight in (0, 1), so they
# cannot revive a ReLU channel that is already zero — a high dead share
# has to be fixed upstream (initialisation, learning rate, activation).
# For over/underfitting, compare the train loss with the validation
# accuracy curve below. load_cifar10 applies NO augmentation here, so
# if the train-val gap is large (say >15%), the prescription is MORE
# regularisation: add augmentation (flip + crop, see 04), weight decay
# or dropout, or stop training earlier.



Plain CNN baseline for comparison (same as 01_simple_cnn.py).


In [ ]:
# Also train SimpleCNN for direct comparison in the same experiment
# Define inline to avoid executing 01_simple_cnn.py as a side effect


class SimpleCNN(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        # Same SimpleCNN as 01 — Conv(3->32)->BN->ReLU->MaxPool, Conv(32->64)->BN->ReLU->MaxPool
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


print("\nTraining SimpleCNN for comparison...")
simple_cnn_compare = SimpleCNN()
simple_losses, simple_accs = train_model(
    simple_cnn_compare,
    "SimpleCNN_compare",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)



### Checkpoint 2: Training converged


In [ ]:
assert len(resnet_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses"
assert resnet_losses[-1] < resnet_losses[0], "ResNetSE loss should decrease"
assert resnet_accs[-1] > 0.4, (
    f"ResNetSE val accuracy {resnet_accs[-1]:.3f} too low -- "
    "expected > 0.4 on full CIFAR-10"
)

# Architecture comparison table
print(f"\n{'=' * 50}")
print(f"  ARCHITECTURE COMPARISON")
print(f"{'=' * 50}")
print(f"{'Model':>15} {'Params':>10} {'Final Loss':>12} {'Val Acc':>10}")
print("-" * 50)
print(
    f"{'SimpleCNN':>15} {count_parameters(simple_cnn_compare):>10,} "
    f"{simple_losses[-1]:>12.4f} {simple_accs[-1]:>9.3f}"
)
print(
    f"{'ResNetSE':>15} {resnet_params:>10,} "
    f"{resnet_losses[-1]:>12.4f} {resnet_accs[-1]:>9.3f}"
)

improvement = resnet_accs[-1] - simple_accs[-1]
print(
    f"\n  ResNetSE improvement: {improvement:+.3f} ({improvement/simple_accs[-1]:+.1%} relative)"
)

# INTERPRETATION: ResNetSE should achieve higher accuracy than SimpleCNN.
# The skip connections let gradients flow directly through the network,
# and the SE block re-weights channels so the most informative feature
# maps get amplified. The improvement is modest on shallow networks but
# becomes dramatic as depth increases (ResNet-50 vs VGG-19, for example).
print("\n--- Checkpoint 2 passed --- ResNetSE trained and compared\n")

# Register in ModelRegistry
if has_registry:
    # TODO: Register the trained ResNetSE model with register_model under the
    #   name "resnet_se_cifar10", with its final-epoch loss and val accuracy
    resnet_version = ____

# Save comparison plots
viz = create_visualizer()
# TODO: Save loss and accuracy comparison plots with save_training_plots
#   One call per metric, each overlaying BOTH models' curves in one dict
#   (e.g. keys "SimpleCNN loss" / "ResNetSE loss").
#   Files: "ex_2_02_arch_comparison_loss.html" (Training Loss) and
#          "ex_2_02_arch_comparison_acc.html" (Validation Accuracy)
____
____



## PHASE 4 — VISUALISE: Grad-CAM Heatmaps


Args:
        model: The CNN model
        input_tensor: Input image (1, C, H, W)
        target_class: Class index to explain
        target_layer: Conv layer to compute CAM for

    Returns:
        Heatmap as numpy array (H, W), values in [0, 1]


In [ ]:
# Grad-CAM (Gradient-weighted Class Activation Mapping) answers:
#   "WHERE in the image did the model look to make its decision?"
#
# It works by:
#   1. Run a forward pass for the target class
#   2. Compute gradients of the target class score w.r.t. the last
#      convolutional layer's feature maps
#   3. Weight each feature map by the average gradient (how important
#      is this feature map for this class?)
#   4. Sum the weighted feature maps and apply ReLU (keep only
#      positive contributions)
#   5. Overlay the heatmap on the original image
#
# This is the gold standard for CNN interpretability. When a medical
# imaging model says "this X-ray shows pneumonia", Grad-CAM shows
# WHETHER the model looked at the lungs (good) or the patient ID label
# on the corner of the film (bad — a known failure mode).

print("=" * 70)
print("  PHASE 4 — VISUALISE: Grad-CAM Heatmaps (Where Does the Model Look?)")
print("=" * 70)


def compute_gradcam(
    model: nn.Module,
    input_tensor: torch.Tensor,
    target_class: int,
    target_layer: nn.Module,
) -> np.ndarray:
    gradients = []
    activations = []

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    def forward_hook(module, input, output):
        activations.append(output.detach())

    bh = target_layer.register_full_backward_hook(backward_hook)
    fh = target_layer.register_forward_hook(forward_hook)

    model.eval()
    model.zero_grad()

    # TODO: Forward pass for Grad-CAM, then pick the target class's logit
    #   1. output: the model's logits for input_tensor (batch of 1)
    #   2. target_score: the scalar logit for target_class (the backward
    #      pass below differentiates this score, not the loss)
    output = ____
    target_score = ____
    target_score.backward()

    bh.remove()
    fh.remove()

    # TODO: Compute the Grad-CAM heatmap
    #   1. grads, acts: the hooked gradient and activation for the target
    #      layer, with the batch dim dropped — each shape (C, H, W)
    #   2. weights: one importance number per channel = the spatial mean of
    #      that channel's gradient — shape (C,)
    #   3. cam: sum over channels of (channel weight x activation map) —
    #      shape (H, W); reshape weights so they broadcast over H and W
    #   (ReLU, normalisation to [0, 1] and upsampling are done for you below)
    grads = ____
    acts = ____
    weights = ____
    cam = ____
    cam = F.relu(cam)

    if cam.max() > 0:
        cam = cam / cam.max()

    cam = F.interpolate(
        cam.unsqueeze(0).unsqueeze(0),
        size=(input_tensor.shape[2], input_tensor.shape[3]),
        mode="bilinear",
        align_corners=False,
    ).squeeze()

    return cam.numpy()


# Generate Grad-CAM for several validation images
sample_indices = [0, 42, 100, 500, 1000, 2000, 3000, 5000]

# Use the last conv layer in the second ResBlock
target_layer = resnet_se.block2.conv2

fig_gradcam, axes = plt.subplots(3, len(sample_indices), figsize=(24, 10))
fig_gradcam.suptitle(
    "Grad-CAM: Where Does ResNetSE Look? (Red = High Attention)",
    fontsize=14,
)

for col, idx in enumerate(sample_indices):
    img = X_val[idx : idx + 1].clone().requires_grad_(True)
    true_label = y_val[idx].item()
    true_name = CLASS_NAMES[true_label]

    # Get prediction
    with torch.no_grad():
        logits = resnet_se(X_val[idx : idx + 1])
        pred_label = logits.argmax(dim=-1).item()
        pred_name = CLASS_NAMES[pred_label]
        conf = F.softmax(logits, dim=-1)[0, pred_label].item()

    # TODO: Compute the Grad-CAM heatmap for the PREDICTED class using
    #   compute_gradcam on resnet_se at target_layer
    heatmap = ____

    # Row 0: Original image
    orig = denormalise_cifar(X_val[idx])
    axes[0, col].imshow(orig.permute(1, 2, 0).numpy())
    correct = pred_label == true_label
    colour = "green" if correct else "red"
    axes[0, col].set_title(f"True: {true_name}", fontsize=8)
    axes[0, col].axis("off")

    # Row 1: Grad-CAM heatmap
    axes[1, col].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
    axes[1, col].set_title(f"Pred: {pred_name} ({conf:.0%})", fontsize=8, color=colour)
    axes[1, col].axis("off")

    # Row 2: Overlay (heatmap on original)
    orig_np = orig.permute(1, 2, 0).numpy()
    heatmap_colour = plt.cm.jet(heatmap)[:, :, :3]
    overlay = 0.5 * orig_np + 0.5 * heatmap_colour
    overlay = np.clip(overlay, 0, 1)
    axes[2, col].imshow(overlay)
    axes[2, col].set_title("Overlay", fontsize=8)
    axes[2, col].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Grad-CAM", fontsize=10)
axes[2, 0].set_ylabel("Overlay", fontsize=10)

plt.tight_layout()
plt.savefig("ex_2_02_gradcam.png", dpi=150, bbox_inches="tight")
plt.close(fig_gradcam)
print("  Saved: ex_2_02_gradcam.png")
print("  Red regions = where the model focused to make its prediction")
print("  Check: does the model look at the object or the background?")

# SE attention weight analysis
print("\n  SE ATTENTION WEIGHT ANALYSIS:")
resnet_se.eval()
se_weights_per_class: dict[str, list[np.ndarray]] = {name: [] for name in CLASS_NAMES}

with torch.no_grad():
    se_outputs = []

    def se_hook(module, input, output):
        b, c, _, _ = input[0].shape
        s = F.adaptive_avg_pool2d(input[0], 1).view(b, c)
        w = module.fc(s)
        se_outputs.append(w.cpu().numpy())

    hook_handle = resnet_se.se1.register_forward_hook(se_hook)

    for batch_start in range(0, min(2000, len(X_val)), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, 2000)
        _ = resnet_se(X_val[batch_start:batch_end])

    hook_handle.remove()

    all_se_weights = np.concatenate(se_outputs, axis=0)
    for i in range(min(2000, len(y_val))):
        label = CLASS_NAMES[y_val[i].item()]
        se_weights_per_class[label].append(all_se_weights[i])

# Show average SE weights per class
print(f"  {'Class':>12s}  Top-3 channels (most attended)")
print("  " + "-" * 50)
for cls_name in CLASS_NAMES:
    if se_weights_per_class[cls_name]:
        avg_weights = np.mean(se_weights_per_class[cls_name], axis=0)
        top3 = np.argsort(avg_weights)[-3:][::-1]
        top3_str = ", ".join(f"ch{c}({avg_weights[c]:.2f})" for c in top3)
        print(f"  {cls_name:>12s}  {top3_str}")



### Checkpoint 3: Visualisations generated


In [ ]:
import os

assert os.path.exists("ex_2_02_gradcam.png"), "Grad-CAM visualisation missing"
assert os.path.exists("ex_2_02_arch_comparison_loss.html"), "Loss comparison missing"
print("\n--- Checkpoint 3 passed --- Grad-CAM and SE analysis complete\n")



## PHASE 5 — APPLY: Semiconductor Wafer Inspection at a Singapore Fab


WAFER INSPECTION SIMULATION ({total:,} inspection images):

  Detection Performance:
    Defects correctly caught:  {caught:>5,} / {actual_defects:,} ({detection_rate:.1%} detection rate)
    Defects missed (CRITICAL): {missed:>5,} / {actual_defects:,} ({1-detection_rate:.1%} miss rate)
    False alarms:              {false_alarm:>5,} / {actual_passes:,} ({false_alarm_rate:.1%} false alarm rate)
    Correct passes:            {correct_pass:>5,} / {actual_passes:,}

  Financial Impact (illustrative fab production volume):
    Monthly wafer throughput:         ~10,000 wafers
    Monthly inspection images:        ~200,000

    COST OF MISSED DEFECTS (this simulation):
      Missed defects: {missed:,} at avg ${inspection_results["missed_cost"]/max(missed,1):,.0f}/defect
      Total missed defect cost: ${inspection_results["missed_cost"]:>12,.0f}

    COST COMPARISON (monthly, at production scale):
      Human inspection (15 inspectors x 3 shifts):  $  202,500
      CNN system (compute + 5 edge-case inspectors): $   80,000
      CNN-missed defect scrap (estimated):           $   45,000
      Net monthly savings:                           $   77,500
      Projected annual savings:                      $  930,000

  WHY GRAD-CAM IS CRITICAL HERE:
    When the CNN flags a defect, the inspector needs to know WHERE.
    Without Grad-CAM: inspector re-scans entire 512x512 image (~30 sec)
    With Grad-CAM: inspector zooms to highlighted region (~5 sec)
    Time savings: 83% faster verification per flagged image

  STAKEHOLDER-READY OUTPUT:
    "The CNN inspection system detects {detection_rate:.0%} of wafer defects
    automatically, with a {false_alarm_rate:.0%} false alarm rate.
    Grad-CAM heatmaps show inspectors exactly where each defect is,
    reducing verification time from 30 seconds to 5 seconds.
    Projected annual savings: $930K from reduced headcount + $2-5M
    from catching fatigue-related misses in late-shift inspection."


In [ ]:
# SCENARIO: You are an ML engineer at a Singapore semiconductor fab
# (Woodlands). The fab produces 300mm wafers for automotive, IoT, and
# 5G chips. Each wafer goes through 500+ processing steps over 3 months.
# Defects at ANY step can scrap the entire wafer ($5,000-$50,000 each).
#
# CURRENT PROCESS:
#   - Automated Optical Inspection (AOI) machines capture high-res
#     images at 20+ inspection points per wafer
#   - Each image is reviewed by a human inspector who classifies:
#     PASS, PARTICLE, SCRATCH, PATTERN_DEFECT, CONTAMINATION
#   - 15 inspectors per shift, 3 shifts/day, ~2,000 images/inspector/day
#   - Human accuracy: ~92% (fatigue causes misses in late-shift hours)
#   - False negative cost: $5,000-$50,000 (defective wafer continues
#     through remaining processing steps, wasting all downstream work)
#   - False positive cost: $200-$500 (unnecessary re-inspection)
#
# WHY RESNET + SE + GRAD-CAM:
#   1. ResNet handles high-resolution inspection images (512x512+)
#      without the degradation problem
#   2. SE attention learns which feature channels are important for
#      each defect type (scratches activate edge channels; particles
#      activate texture channels; contamination activates colour channels)
#   3. Grad-CAM provides EXPLAINABILITY: when the model flags a defect,
#      it shows WHERE on the wafer the defect was detected. Inspectors
#      can verify in seconds instead of re-scanning the entire image.
#
# BUSINESS CASE:
#   Manual inspection: 15 inspectors * 3 shifts * $4,500/month = $202,500/month
#   CNN system: $30,000/month compute + $50,000/month (5 inspectors for edge cases)
#   Monthly savings: $122,500
#   Annual savings: $1.47M
#   Additional: CNN catches late-shift fatigue misses -> estimated $2-5M
#   in avoided scrap per year

print("=" * 70)
print("  PHASE 5 — APPLY: Semiconductor Wafer Inspection (Singapore fab)")
print("=" * 70)

# TODO: Build the wafer inspection simulation
#   1. Create WAFER_MAPPING: airplane/ship -> "PASS", automobile/truck -> "PATTERN_DEFECT",
#      bird/horse -> "PARTICLE", cat/dog -> "SCRATCH", deer/frog -> "CONTAMINATION"
#   2. Create DEFECT_COST: PASS=0, PARTICLE=15000, SCRATCH=25000,
#      PATTERN_DEFECT=8000, CONTAMINATION=35000
#   3. Run inference and classify results
WAFER_MAPPING = {____}

DEFECT_COST = {____}

resnet_se.eval()
with torch.no_grad():
    val_logits = resnet_se(X_val)
    val_probs = F.softmax(val_logits, dim=-1)
    val_preds = val_logits.argmax(dim=-1)
    val_confidences = val_probs.gather(1, val_preds.unsqueeze(1)).squeeze()

# TODO: Classify each image as wafer inspection result
#   Loop through all validation images, map CIFAR predictions to wafer categories,
#   count: correct_defect_caught, missed_defect, false_alarm, correct_pass
#   Track missed_cost and false_alarm_cost
inspection_results = {
    "total": 0,
    "correct_defect_caught": 0,
    "missed_defect": 0,
    "false_alarm": 0,
    "correct_pass": 0,
    "missed_cost": 0.0,
    "false_alarm_cost": 0.0,
}

INSPECTION_THRESHOLD = 0.70

for i in range(len(y_val)):
    true_cifar = CLASS_NAMES[y_val[i].item()]
    pred_cifar = CLASS_NAMES[val_preds[i].item()]
    true_wafer = WAFER_MAPPING[true_cifar]
    pred_wafer = WAFER_MAPPING[pred_cifar]

    inspection_results["total"] += 1

    # TODO: Implement the classification logic
    #   if true is defect AND pred is defect -> correct_defect_caught
    #   if true is defect AND pred is PASS -> missed_defect (add DEFECT_COST[true_wafer])
    #   if true is PASS AND pred is defect -> false_alarm (add $300 re-inspection)
    #   else -> correct_pass
    ____

total = inspection_results["total"]
caught = inspection_results["correct_defect_caught"]
missed = inspection_results["missed_defect"]
false_alarm = inspection_results["false_alarm"]
correct_pass = inspection_results["correct_pass"]

actual_defects = caught + missed
actual_passes = correct_pass + false_alarm
detection_rate = caught / actual_defects if actual_defects > 0 else 0
false_alarm_rate = false_alarm / actual_passes if actual_passes > 0 else 0

print(
)



### Checkpoint 4: Apply section complete


In [ ]:
assert inspection_results["total"] == len(y_val), "Should inspect all samples"
assert (
    detection_rate > 0.3
), f"Detection rate {detection_rate:.3f} too low -- model should catch some defects"
print("--- Checkpoint 4 passed --- wafer inspection application demonstrated\n")



## Clean up


In [ ]:
await conn.close()



## REFLECTION


THEORY:
  [x] The degradation problem: deeper is not always better without
      skip connections (the "telephone game" problem)
  [x] Residual learning: y = F(x) + x -- learn the residual, not the
      full transformation. Gradients flow directly through shortcuts.
  [x] SE attention: squeeze (pool) -> excite (MLP) -> scale (re-weight)
      Learns which channels matter for each specific input.

  BUILD + TRAIN:
  [x] ResBlock: two-conv residual block with BatchNorm
  [x] SEBlock: channel attention with <1% parameter overhead
  [x] ResNetSE: {resnet_params:,} params, {resnet_accs[-1]:.1%} val accuracy
  [x] Compared against SimpleCNN: {improvement:+.3f} accuracy improvement

  VISUALISE (the proof):
  [x] Grad-CAM heatmaps: WHERE the model looks for each prediction
  [x] SE attention weights: WHICH channels each class relies on
  [x] Overlay visualisation: heatmap on original for instant verification

  APPLY:
  [x] Singapore-fab semiconductor wafer inspection
  [x] Detection rate: {detection_rate:.0%} of defects caught automatically
  [x] Grad-CAM enables 83% faster inspector verification (30s -> 5s)
  [x] Projected annual savings: $930K + $2-5M avoided scrap

  KEY INSIGHT: Residual connections and SE attention are not academic
  curiosities -- they are production requirements. ResNets enabled the
  first superhuman image classifiers. SE attention costs almost nothing
  but tells you WHICH features the model cares about. And Grad-CAM
  turns a "black box" classifier into an explainable decision that
  inspectors, regulators, and stakeholders can verify visually.

  Next: In 03_production_pipeline.py, you'll export the best model to
  ONNX and serve it through InferenceServer -- the bridge from
  "it works in a notebook" to "it works in production"...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

